In [ ]:
import math
import os
from typing import Optional, Tuple, Union

import numpy as np
import pandas as pd

from scipy import stats as st
SCIPY_AVAILABLE = True

In [ ]:
def parse_col_selector(df: pd.DataFrame, selector: Union[str, int]) -> str:
    if isinstance(selector, int):
        if selector < 0 or selector >= df.shape[1]:
            raise ValueError(f"Column index {selector} out of bounds for {df.shape[1]} columns.")
        return df.columns[selector]

    sel = str(selector).strip()

    if sel in df.columns:
        return sel

    if sel.isdigit():
        idx = int(sel)
        if idx < 0 or idx >= df.shape[1]:
            raise ValueError(f"Column index {idx} out of bounds for {df.shape[1]} columns.")
        return df.columns[idx]

    lower_map = {c.lower(): c for c in df.columns}
    if sel.lower() in lower_map:
        return lower_map[sel.lower()]

    if sel.isalpha():
        up = sel.upper()
        idx = 0
        for ch in up:
            idx = idx * 26 + (ord(ch) - ord('A') + 1)
        idx -= 1  # zero-based
        if idx < 0 or idx >= df.shape[1]:
            raise ValueError(f"Excel-like selector '{selector}' maps to index {idx}, which is out of bounds for {df.shape[1]} columns.")
        return df.columns[idx]

    raise ValueError(f"Cannot resolve column selector '{selector}'. Available columns: {list(df.columns)}")

In [ ]:
def load_series_from_csv(path: str, date_col_sel: Union[str, int], qlike_col_sel: Union[str, int]) -> pd.DataFrame:
    df = pd.read_csv(path)
    date_col = parse_col_selector(df, date_col_sel)
    qlike_col = parse_col_selector(df, qlike_col_sel)

    out = pd.DataFrame({
        'Date': pd.to_datetime(df[date_col], errors='coerce'),
        'QLIKE': pd.to_numeric(df[qlike_col], errors='coerce'),
    })
    out = out.dropna(subset=['Date', 'QLIKE']).copy()
    out = out.drop_duplicates(subset=['Date']).sort_values('Date').set_index('Date')
    return out

In [ ]:
def hac_long_run_variance(x: np.ndarray, lags: Optional[int]) -> float:
    n = x.shape[0]
    if n < 2:
        return float('nan')

    if lags is None or lags < 0:
        lags = 0
    lags = min(lags, n - 1)

    x = x - x.mean()
    gamma0 = np.dot(x, x) / n
    S = gamma0

    for k in range(1, lags + 1):
        w_k = 1.0 - k / (lags + 1.0)
        cov = np.dot(x[k:], x[:-k]) / n
        S += 2.0 * w_k * cov

    return float(S)

In [ ]:
def diebold_mariano(loss1: np.ndarray,
                    loss2: np.ndarray,
                    h: int = 1,
                    lags: Union[str, int, None] = 'auto',
                    use_hln: bool = True) -> Tuple[float, float, float, float, float, float]:
    loss1 = np.asarray(loss1, dtype=float)
    loss2 = np.asarray(loss2, dtype=float)
    if loss1.shape != loss2.shape:
        raise ValueError('loss1 and loss2 must have the same shape after alignment.')

    d = loss1 - loss2  # positive -> model2 better (lower loss)
    n = d.size
    if n < 5:
        raise ValueError('Not enough overlapping observations to run the DM test (need at least 5).')
    mean_d = float(np.mean(d))

    # Lags default
    if isinstance(lags, str):
        if lags.lower() == 'auto':
            lags_val = max(h - 1, 0)
        else:
            raise ValueError("lags must be 'auto', an integer, or None")
    elif lags is None:
        lags_val = max(h - 1, 0)
    else:
        lags_val = int(lags)
        if lags_val < 0:
            raise ValueError('lags must be >= 0')

    # HAC long-run variance
    d_centered = d - mean_d
    S = hac_long_run_variance(d_centered, lags=lags_val)
    if not np.isfinite(S) or S <= 0:
        raise ValueError('Long-run variance estimate is non-positive or NaN; cannot compute DM statistic.')

    # DM (asymptotic normal)
    var_mean = S / n
    dm = mean_d / math.sqrt(var_mean)

    # Two-sided p-values
    if 'SCIPY_AVAILABLE' in globals() and SCIPY_AVAILABLE:
        p_dm = 2.0 * (1.0 - st.norm.cdf(abs(dm)))  # type: ignore
    else:
        p_dm = 2.0 * (1.0 - _normal_cdf(abs(dm)))

    # HLN small-sample adjustment
    adj = math.sqrt((n + 1 - 2 * h + (h * (h - 1)) / n) / n)
    dm_hln = dm * adj
    if 'SCIPY_AVAILABLE' in globals() and SCIPY_AVAILABLE:
        p_dm_hln = 2.0 * (1.0 - st.t.cdf(abs(dm_hln), df=n - 1))  # type: ignore
    else:
        p_dm_hln = 2.0 * (1.0 - _t_cdf(abs(dm_hln), df=n - 1))

    return dm, p_dm, dm_hln, p_dm_hln, mean_d, S

In [ ]:
def run_dm(file1: str,
           file2: str,
           date_col: Union[str, int] = 'B',
           qlike_col: Union[str, int] = 'I',
           h: int = 1,
           lags: Union[str, int, None] = 'auto',
           alpha: float = 0.05,
           use_hln: bool = True) -> str:
    df1 = load_series_from_csv(file1, date_col, qlike_col).rename(columns={'QLIKE': 'QLIKE_1'})
    df2 = load_series_from_csv(file2, date_col, qlike_col).rename(columns={'QLIKE': 'QLIKE_2'})

    merged = df1.join(df2, how='inner').dropna()
    n = merged.shape[0]
    if n < 5:
        raise ValueError(f'After aligning on Date, only {n} rows remain; need at least 5 to run DM test.')

    dm, p_dm, p_dm_hln_stat, p_dm_hln_p, mean_diff, lrv = diebold_mariano(
        merged['QLIKE_1'].values,
        merged['QLIKE_2'].values,
        h=h,
        lags=lags,
        use_hln=use_hln
    )

    # Unpack properly (fix variable names)
    dm_stat, p_dm, dm_hln, p_dm_hln, mean_diff, lrv = dm, p_dm, p_dm_hln_stat, p_dm_hln_p, mean_diff, lrv

    avg1 = float(merged['QLIKE_1'].mean())
    avg2 = float(merged['QLIKE_2'].mean())

    # Decide winner based on mean diff (remember: lower QLIKE is better)
    better = None
    if mean_diff > 0:
        better = os.path.basename(file2)
    elif mean_diff < 0:
        better = os.path.basename(file1)

    stat_used = dm_hln if use_hln else dm_stat
    p_used = p_dm_hln if use_hln else p_dm
    stat_name = 'DM (HLN adjusted)' if use_hln else 'DM (asymptotic normal)'

    lines = []
    lines.append('Diebold–Mariano Test for Predictive Accuracy (QLIKE loss)')
    lines.append('-' * 72)
    lines.append('Files:')
    lines.append(f'  Model 1: {file1}')
    lines.append(f'  Model 2: {file2}')
    lines.append('Columns used:')
    lines.append(f'  Date column: {date_col}')
    lines.append(f'  QLIKE column: {qlike_col}')
    lines.append(f'Forecast horizon h: {h}')
    lines.append(f"HAC lags: {('auto -> ' + str(max(h-1,0))) if (isinstance(lags,str) and lags=='auto') else str(lags)}")
    lines.append('')
    lines.append(f'Aligned sample size (after inner join on Date): n = {n}')
    lines.append('Average QLIKE:')
    lines.append(f'  {os.path.basename(file1)}: {avg1:.6g}')
    lines.append(f'  {os.path.basename(file2)}: {avg2:.6g}')
    lines.append('')
    lines.append('Test statistics:')
    lines.append(f'  DM (normal): {dm_stat:.4f}  |  two-sided p = {p_dm:.4g}')
    lines.append(f'  DM (HLN):    {dm_hln:.4f} |  two-sided p = {p_dm_hln:.4g}')
    lines.append('')
    if p_used < alpha:
        if better is None:
            lines.append(f'Conclusion @ alpha={alpha}: Significant difference, but mean losses equal within rounding.')
        else:
            lines.append(f'Conclusion @ alpha={alpha}: {better} has significantly LOWER QLIKE than the other (p = {p_used:.4g}, using {stat_name}).')
    else:
        lines.append(f'Conclusion @ alpha={alpha}: No statistically significant difference (p = {p_used:.4g}, using {stat_name}).')
        if better is not None:
            lines.append(f'(Point estimate favors {better}, but not significantly at alpha={alpha}.)')

    return "\n".join(lines)

In [ ]:
#Parameters
file1 = "LSTM.csv"       # e.g., "MLP.csv"
file2 = "LSTM BIC.csv"   # e.g., "MLP BIC.csv"
date_col = "B"          # can be a name, index, or Excel letter (e.g., "B")
qlike_col = "I"         # can be a name, index, or Excel letter (e.g., "I")
h = 1                   # forecast horizon
lags = "auto"           # 'auto' uses h-1; or set a non-negative integer
alpha = 0.05            # significance level
use_hln = True          # use HLN adjustment for the conclusion

# Run the test
print(run_dm(file1, file2, date_col=date_col, qlike_col=qlike_col, h=h, lags=lags, alpha=alpha, use_hln=use_hln))